# Parameter Grid Reliability: Blur × Motion Censoring
- Extends `main.ipynb` to the parameter grid run by `x07.blur_censor_param.csh` (`coordinatesAll_param.csv`).
- Compares reliability (mean pairwise distance between sessions) across smoothing kernel size (FWHM) and motion censor limit, then tests whether they interact.

In [ ]:
import os
import pandas as pd
import itertools
import numpy as np

from scipy.spatial.distance import pdist
from scipy.stats import ttest_rel
from statsmodels.stats.anova import AnovaRM

# Plotting Packages
import seaborn as sns
import matplotlib.pyplot as plt
import ptitprince as pt # Only needed for raincloud plot
import configFunctions as config

# Variables
dataFile = 'coordinatesAll_param.csv' # Path to the CSV file containing the parameter grid coordinates

# Peak search to analyze: 'abs' matches main.ipynb (|z| peak); 'pos' / 'neg' are signed peaks.
signSel = 'abs'

# Only keep sessions present in every grid cell, so each mean distance is built from the same sessions.
matchSessions = True

# Output directory
output_dir = 'output/paramGrid'
os.makedirs(output_dir, exist_ok=True)

### Required files
- `coordinatesAll_param.csv` - generated by serverFunctions, rows per variant identified by `runTag` (e.g. `param-b4m0p3` = FWHM 4 mm, motion limit 0.3).

In [ ]:
# A configuration function to make output images easier to modify for publication.
config.svg_editing()

In [ ]:
# load data, inspect (subject/session as strings so leading zeros survive)
df = pd.read_csv(dataFile, dtype={'subject': str, 'session': str})
df.head()

### Intermediate clean up
- Rows without a `runTag` are from runs outside the grid, and the base `param` run is the same settings as `param-b4m0p3`, so both are dropped.
- Blur and motion limit are parsed from the tag: `b{F}m{M}`, with `p` standing in for `.`.

In [ ]:
df = df[df['sign'] == signSel]
df = df[df['runTag'].str.contains(r'-b\d', na=False)].copy()

tagPattern = r'-b(?P<blur>[\dp]+)m(?P<motion>[\dp]+)$'
parsed = df['runTag'].str.extract(tagPattern)
df['blur'] = parsed['blur'].str.replace('p', '.').astype(float)
df['motion'] = parsed['motion'].str.replace('p', '.').astype(float)

print(f"sign = '{signSel}': {len(df)} rows")
print(pd.crosstab(df['blur'], df['motion']))

In [ ]:
# Pull out and show the rows which include any nans
rows_with_nan = df[df[['X', 'Y', 'Z']].isna().any(axis=1)]
rows_with_nan.to_csv(os.path.join(output_dir, 'coordinatesParam_NaNRows.csv'), index=False)
subj_with_nan = rows_with_nan['subject'].unique()
print(f'Subjects with NaNs - {len(subj_with_nan)}')
print(f'List: {subj_with_nan}')

df = df[~df['subject'].isin(subj_with_nan)]

# Sessions missing from some grid cells (e.g. a motion limit skipped for too few TRs)
cellVars = ['blur', 'motion', 'target', 'clustMask']
nCells = df[cellVars].drop_duplicates().shape[0]
sesCells = df.groupby(['subject', 'session'])[cellVars].apply(lambda g: len(g.drop_duplicates()))
partialSes = sesCells[sesCells < nCells]
print(f'\nSessions missing grid cells ({nCells} expected):')
print(partialSes.to_string() if len(partialSes) else '  none')

if matchSessions and len(partialSes):
    dropIdx = pd.MultiIndex.from_frame(df[['subject', 'session']]).isin(partialSes.index)
    df = df[~dropIdx]
    print(f'Dropped {dropIdx.sum()} rows from those sessions')

print(f"\n{len(df['subject'].unique())} Subjects Remaining...")

In [ ]:
# Span the set of each combination of parameters, take the mean euclidean distance across sessions.
grouping_vars = ['subject', 'blur', 'motion', 'target', 'clustMask']

combodf = (df.groupby(grouping_vars)[['X', 'Y', 'Z']]
             .apply(lambda g: round(pdist(g.values, metric='euclidean').mean(), 2))
             .rename('mean_distance')
             .reset_index())

# AnovaRM needs every subject in every cell
nPerSubj = combodf.groupby('subject').size()
incomplete = nPerSubj[nPerSubj < nPerSubj.max()].index
if len(incomplete):
    print(f'Dropping {len(incomplete)} subjects with incomplete cells: {list(incomplete)}')
    combodf = combodf[~combodf['subject'].isin(incomplete)]

combodf.to_csv(os.path.join(output_dir, 'combodf_param.csv'), index=False)
print(f"{combodf['subject'].nunique()} subjects x {nPerSubj.max()} cells")
combodf.head()

# Plotting

In [ ]:
targetName = 'sgACC' # For plotting, what does the 'target' mask represent?
clusterRegion = 'dlPFC' # for plotting, what does the 'clusterMask' mask represent?

# Grid factors (this notebook's focus); target and clustMask are carried over from main.ipynb
gridVars = ['blur', 'motion']
gridVarNames = ['Smoothing Kernel (FWHM)', 'Motion Censor Limit']

orderDict = {
    'blur': sorted(combodf['blur'].unique()),
    'motion': sorted(combodf['motion'].unique()),
    'target': ['seed', 'network'],
    'clustMask': ['erode_1', 'orig', 'dilate_5'],
}
nameDict = {
    'blur': [f'{b:g} mm' if b else 'None' for b in orderDict['blur']],
    'motion': [f'{m:g} mm' for m in orderDict['motion']],
    'target': ['Seed', 'Network'],
    'clustMask': ['Eroded', 'Original', 'Dilated'],
}
varNameDict = {'blur': 'Smoothing Kernel (FWHM)', 'motion': 'Motion Censor Limit',
               'target': f'{targetName} mask', 'clustMask': f'{clusterRegion} mask'}

ylabel = 'Mean Pairwise Distance (mm)'

In [ ]:
def plot_significance_brackets(ax, comparisons, y_min, y_max):
    """
    Plot significance brackets and stars on an axis.

    Parameters:
    -----------
    ax : matplotlib axis
        The axis to plot on
    comparisons : list of dict
        List of comparisons, each containing 'pos1', 'pos2', and 'pval'
    y_min : float
        Minimum y-value in the data
    y_max : float
        Maximum y-value in the data

    Returns:
    --------
    tuple : (new_y_min, new_y_max) adjusted limits to accommodate brackets
    """
    y_range = y_max - y_min

    if not comparisons:
        return (y_min - 0.05 * y_range, y_max + 0.05 * y_range)

    bracket_height_start = y_max + 0.02 * y_range
    bracket_spacing = 0.08 * y_range

    for comp_idx, comp in enumerate(comparisons):
        x1 = comp['pos1']
        x2 = comp['pos2']
        pval = comp['pval']

        # Determine star based on p-value
        if pval < 0.001:
            star = '***'
        elif pval < 0.01:
            star = '**'
        elif pval < 0.05:
            star = '*'
        else:
            star = 'ns'

        # Calculate bracket height (stack them)
        bracket_y = bracket_height_start + (comp_idx * bracket_spacing)
        star_y = bracket_y + 0.002 * y_range

        # Draw horizontal line
        ax.plot([x1, x2], [bracket_y, bracket_y], 'k-', linewidth=1.5)

        # Draw vertical ticks on both ends
        tick_height = 0.015 * y_range
        ax.plot([x1, x1], [bracket_y - tick_height, bracket_y], 'k-', linewidth=1.5)
        ax.plot([x2, x2], [bracket_y - tick_height, bracket_y], 'k-', linewidth=1.5)

        # Add star
        ax.text((x1 + x2) / 2, star_y, star, ha='center', va='bottom',
                fontsize=14, fontweight='bold')

    # Return adjusted y-limits to accommodate all brackets
    max_bracket_y = bracket_height_start + (len(comparisons) - 1) * bracket_spacing
    return (y_min - 0.05 * y_range, max_bracket_y + 0.10 * y_range)

In [ ]:
def rm_test(data, var, groups):
    """
    One-way repeated measures test of mean_distance across levels of var (data has one row per subject x level).
    Paired t-test for 2 levels, RM-ANOVA + Bonferroni-corrected paired t-tests for more.
    Returns a dict in the same layout as main.ipynb's stat_results entries, plus the full 'pairwise' list.
    """
    data = data.sort_values('subject')
    group_data = [data[data[var] == val]['mean_distance'].values for val in groups]

    if len(groups) == 2:
        stat, p_val = ttest_rel(group_data[0], group_data[1])
        return {'test': 'ttest', 'stat': stat, 'p_val': p_val, 'stat_name': 't', 'groups': groups,
                'pairwise': [], 'comparisons': [{'group1': groups[0], 'group2': groups[1],
                                                 'pval': p_val, 'pos1': 0, 'pos2': 1}]}

    res = AnovaRM(data=data, depvar='mean_distance', subject='subject', within=[var]).fit()
    stat = res.anova_table['F Value'].values[0]
    p_val = res.anova_table['Pr > F'].values[0]

    n_comparisons = len(groups) * (len(groups) - 1) // 2
    pairwise_results = []
    for g1, g2 in itertools.combinations(range(len(groups)), 2):
        t_stat, t_pval = ttest_rel(group_data[g1], group_data[g2])
        corrected_pval = min(t_pval * n_comparisons, 1.0)
        pairwise_results.append({
            'group1': groups[g1], 'group2': groups[g2],
            'meandiff': group_data[g2].mean() - group_data[g1].mean(),
            't_stat': t_stat, 'p_raw': t_pval, 'p_corrected': corrected_pval,
            'reject': corrected_pval < 0.05,
            'pos1': g1, 'pos2': g2,
        })

    # Significant comparisons for plotting, shortest bracket first
    comparisons = [{'group1': r['group1'], 'group2': r['group2'], 'pval': r['p_corrected'],
                    'pos1': r['pos1'], 'pos2': r['pos2']} for r in pairwise_results if r['reject']]
    comparisons.sort(key=lambda x: abs(x['pos2'] - x['pos1']))

    return {'test': 'anova', 'stat': stat, 'p_val': p_val, 'stat_name': 'F', 'groups': groups,
            'pairwise': pairwise_results, 'comparisons': comparisons}


def print_rm_test(result, label):
    if result['test'] == 'ttest':
        print(f"{label}: paired t-test, t={result['stat']:.3f}, p={result['p_val']:.2e}")
        return
    print(f"{label}: RM-ANOVA, F={result['stat']:.3f}, p={result['p_val']:.2e}")
    print('  Post-hoc paired t-tests (Bonferroni corrected):')
    for r in result['pairwise']:
        sig = '*' if r['reject'] else ''
        print(f"    {r['group1']:>6g} vs {r['group2']:<6g}: diff={r['meandiff']:>7.3f}, t={r['t_stat']:>7.3f}, "
              f"p_raw={r['p_raw']:.2e}, p_corr={r['p_corrected']:.2e} {sig}")


def raincloud(ax, data, var, order, names, result, title):
    """Raincloud of mean_distance by var, with the test result under the title and significance brackets."""
    data = data.copy()
    data[var] = pd.Categorical(data[var], categories=order, ordered=True)

    pt.RainCloud(data=data, x=var, y='mean_distance', ax=ax,
        palette='Set2', hue=var, bw=0.3, width_viol=0.7, width_box=0.3,
        orient='v', alpha=0.6, dodge=False, pointplot=False, point_size=3,
        linewidth=2, jitter=0.15, box_showfliers=False, box_linewidth=2)

    ax.set_xticklabels(names)
    if ax.get_legend():
        ax.get_legend().remove()

    # Open up the plot to ensure the peaks for each distribution are visible.
    percSpread = 0.4
    xmin, xmax = ax.get_xlim()
    ax.set_xlim((xmin - percSpread, xmax + percSpread))

    new_y_min, new_y_max = plot_significance_brackets(
        ax=ax, comparisons=result['comparisons'],
        y_min=data['mean_distance'].min(), y_max=data['mean_distance'].max())
    ax.set_ylim(new_y_min, new_y_max)

    ax.set_title(title, fontsize=14, fontweight='bold', pad=22)
    ax.text(0.5, 1.01, f"{result['stat_name']}={result['stat']:.2f}, p={result['p_val']:.2e}",
            transform=ax.transAxes, ha='center', va='bottom', fontsize=12)
    ax.set_xlabel('', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)

## Main effects: smoothing kernel and motion censor limit
- For each factor, average across all other factors so each subject has one value per level, then run a one-way RM-ANOVA with Bonferroni post-hoc paired t-tests (as in `main.ipynb`).

In [ ]:
stat_results = {}
agg_dfs = {}

for var, varName in zip(gridVars, gridVarNames):
    agg_dfs[var] = combodf.groupby(['subject', var])['mean_distance'].mean().reset_index()
    stat_results[var] = rm_test(agg_dfs[var], var, orderDict[var])
    print_rm_test(stat_results[var], varName)
    print()

### Raincloud plot - main effects

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, var, varName in zip(axes, gridVars, gridVarNames):
    raincloud(ax, agg_dfs[var], var, orderDict[var], nameDict[var], stat_results[var], varName)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'Rainplot_blur_motion.svg'))
plt.show()

# Interaction: smoothing kernel × motion censor limit
- 2-way RM-ANOVA on the subject × blur × motion means (averaged over target and cluster mask).
- 4-way RM-ANOVA on the full design, to check whether either grid factor interacts with the target or cluster mask choice from `main.ipynb`.
- Note: `AnovaRM` reports uncorrected (sphericity-assumed) p-values.

In [ ]:
blurMotion_df = combodf.groupby(['subject', 'blur', 'motion'])['mean_distance'].mean().reset_index()

res_2way = AnovaRM(data=blurMotion_df, depvar='mean_distance', subject='subject',
                   within=['blur', 'motion']).fit()
print('2-Way Repeated Measures ANOVA (blur x motion)')
print('=' * 60)
print(res_2way)
twoway_table = res_2way.anova_table

In [ ]:
res_4way = AnovaRM(data=combodf, depvar='mean_distance', subject='subject',
                   within=['blur', 'motion', 'target', 'clustMask']).fit()
print('4-Way Repeated Measures ANOVA (blur x motion x target x clustMask)')
print('=' * 60)
print(res_4way)
fourway_table = res_4way.anova_table

### Interaction plot
- Cell means ± SEM across subjects. Parallel lines = no interaction; converging/crossing lines = the effect of one factor depends on the other.
- Heatmap shows the same cell means on the grid.

In [ ]:
inter_p = twoway_table.loc['blur:motion', 'Pr > F']
inter_F = twoway_table.loc['blur:motion', 'F Value']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Blur on x, one line per motion limit
sns.pointplot(data=blurMotion_df, x='blur', y='mean_distance', hue='motion', order=orderDict['blur'],
              hue_order=orderDict['motion'], palette='crest', errorbar='se', dodge=0.2,
              capsize=0.1, ax=axes[0])
axes[0].set_xticks(range(len(orderDict['blur'])), nameDict['blur'])
axes[0].set_xlabel(varNameDict['blur'])
handles, _ = axes[0].get_legend_handles_labels()
axes[0].legend(handles, nameDict['motion'], title=varNameDict['motion'])

# Motion on x, one line per blur
sns.pointplot(data=blurMotion_df, x='motion', y='mean_distance', hue='blur', order=orderDict['motion'],
              hue_order=orderDict['blur'], palette='flare', errorbar='se', dodge=0.2,
              capsize=0.1, ax=axes[1])
axes[1].set_xticks(range(len(orderDict['motion'])), nameDict['motion'])
axes[1].set_xlabel(varNameDict['motion'])
handles, _ = axes[1].get_legend_handles_labels()
axes[1].legend(handles, nameDict['blur'], title=varNameDict['blur'])

for ax in axes[:2]:
    ax.set_ylabel(ylabel)

# Cell-mean heatmap
cellMeans = blurMotion_df.pivot_table(index='blur', columns='motion', values='mean_distance', aggfunc='mean')
sns.heatmap(cellMeans, annot=True, fmt='.2f', cmap='rocket_r', ax=axes[2],
            xticklabels=nameDict['motion'], yticklabels=nameDict['blur'],
            cbar_kws={'label': ylabel})
axes[2].set_xlabel(varNameDict['motion'])
axes[2].set_ylabel(varNameDict['blur'])
axes[2].set_title('Cell means', fontsize=14, fontweight='bold')

plt.suptitle(f'Blur x Motion interaction: F={inter_F:.2f}, p={inter_p:.2e}', fontsize=16, fontweight='bold', y=1.04)
# No tight_layout here: the heatmap colorbar needs the constrained layout set by config.svg_editing()
plt.savefig(os.path.join(output_dir, 'Interaction_blur_motion.svg'))
plt.show()

### Raincloud plots - simple effects
- Effect of smoothing kernel at each motion limit (top row), and of motion limit at each kernel size (bottom row).
- Each panel is its own one-way RM-ANOVA with Bonferroni post-hoc tests; useful for reading where an interaction comes from.

In [ ]:
simple_results = {}

nCols = max(len(orderDict['motion']), len(orderDict['blur']))
fig, axes = plt.subplots(2, nCols, figsize=(5 * nCols, 10))

# Row 1: blur within each motion level. Row 2: motion within each blur level.
for row, (testVar, splitVar) in enumerate([('blur', 'motion'), ('motion', 'blur')]):
    for col, level in enumerate(orderDict[splitVar]):
        sub = blurMotion_df[blurMotion_df[splitVar] == level]
        result = rm_test(sub, testVar, orderDict[testVar])
        simple_results[(testVar, splitVar, level)] = result
        print_rm_test(result, f'{varNameDict[testVar]} | {splitVar} = {level:g}')
        title = f"{varNameDict[testVar]}\n{varNameDict[splitVar]} = {nameDict[splitVar][col]}"
        raincloud(axes[row, col], sub, testVar, orderDict[testVar], nameDict[testVar], result, title)
    for col in range(len(orderDict[splitVar]), nCols):
        axes[row, col].set_visible(False)
    print()

# Shared y-limits so panels are comparable
y_lo = min(ax.get_ylim()[0] for ax in axes.flat if ax.get_visible())
y_hi = max(ax.get_ylim()[1] for ax in axes.flat if ax.get_visible())
for ax in axes.flat:
    ax.set_ylim(y_lo, y_hi)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'Rainplot_simple_effects.svg'))
plt.show()

### Interactions with target and cluster mask
- Cell means for each grid factor split by the `main.ipynb` factors, to visualize the grid × target / clustMask terms from the 4-way model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, gridVar in enumerate(gridVars):
    for col, otherVar in enumerate(['target', 'clustMask']):
        aggd = combodf.groupby(['subject', gridVar, otherVar])['mean_distance'].mean().reset_index()
        ax = axes[row, col]
        sns.pointplot(data=aggd, x=gridVar, y='mean_distance', hue=otherVar, order=orderDict[gridVar],
                      hue_order=orderDict[otherVar], palette='Set2', errorbar='se', dodge=0.2,
                      capsize=0.1, ax=ax)
        ax.set_xticks(range(len(orderDict[gridVar])), nameDict[gridVar])
        handles, _ = ax.get_legend_handles_labels()
        ax.legend(handles, nameDict[otherVar], title=varNameDict[otherVar])

        term = f'{gridVar}:{otherVar}'
        ax.set_title(f"{varNameDict[gridVar]} x {varNameDict[otherVar]}\n"
                     f"F={fourway_table.loc[term, 'F Value']:.2f}, p={fourway_table.loc[term, 'Pr > F']:.2e}",
                     fontsize=13)
        ax.set_xlabel(varNameDict[gridVar])
        ax.set_ylabel(ylabel)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'Interaction_grid_x_main.svg'))
plt.show()

### Export statistics to file

In [ ]:
stats_path = os.path.join(output_dir, 'stats_paramGrid.txt')

with open(stats_path, 'w') as f:
    f.write('Parameter Grid Statistical Results Summary\n')
    f.write(f"Data: {dataFile}, sign = '{signSel}', matchSessions = {matchSessions}\n")
    f.write(f"N subjects: {combodf['subject'].nunique()}\n")
    f.write('=' * 90 + '\n\n')

    # --- Main effects ---
    f.write('MAIN EFFECTS (one-way, averaged over other factors)\n')
    f.write('=' * 90 + '\n\n')
    for var in gridVars:
        r = stat_results[var]
        f.write(f"Variable: {varNameDict[var]} ({var})\n")
        f.write(f"Groups: {r['groups']}\n")
        f.write(f"{r['stat_name']} = {r['stat']:.4f}, p = {r['p_val']:.2e}\n")
        f.write('\nGroup descriptives:\n')
        for g in r['groups']:
            vals = agg_dfs[var][agg_dfs[var][var] == g]['mean_distance']
            f.write(f"  {g:<8g}: mean={vals.mean():.3f}, SD={vals.std(ddof=1):.3f}, n={len(vals)}\n")
        if r['pairwise']:
            f.write('\nPost-hoc paired t-tests (Bonferroni corrected):\n')
            for p in r['pairwise']:
                sig = '*' if p['reject'] else ''
                f.write(f"  {p['group1']:>6g} vs {p['group2']:<6g}: diff={p['meandiff']:>7.3f}, t={p['t_stat']:>7.3f}, "
                        f"p_raw={p['p_raw']:.2e}, p_corr={p['p_corrected']:.2e} {sig}\n")
        f.write('\n' + '=' * 90 + '\n\n')

    # --- Factorial models ---
    for label, table in [('2-WAY REPEATED MEASURES ANOVA (blur x motion)', twoway_table),
                         ('4-WAY REPEATED MEASURES ANOVA (blur x motion x target x clustMask)', fourway_table)]:
        f.write(label + '\n')
        f.write('=' * 90 + '\n\n')
        f.write(table.to_string())
        f.write('\n\n')
        sig_effects = table[table['Pr > F'] < 0.05]
        if len(sig_effects) > 0:
            f.write('Significant effects (p < 0.05):\n')
            for effect_name, row in sig_effects.iterrows():
                f.write(f"  {effect_name}: F={row['F Value']:.4f}, p={row['Pr > F']:.2e}\n")
        else:
            f.write('No significant effects at p < 0.05.\n')
        f.write('\n' + '=' * 90 + '\n\n')

    # --- Simple effects ---
    f.write('SIMPLE EFFECTS (one-way within each level of the other grid factor)\n')
    f.write('=' * 90 + '\n\n')
    for (testVar, splitVar, level), r in simple_results.items():
        f.write(f"{testVar} | {splitVar} = {level:g}: {r['stat_name']} = {r['stat']:.4f}, p = {r['p_val']:.2e}\n")
        for p in r['pairwise']:
            sig = '*' if p['reject'] else ''
            f.write(f"    {p['group1']:>6g} vs {p['group2']:<6g}: diff={p['meandiff']:>7.3f}, "
                    f"p_corr={p['p_corrected']:.2e} {sig}\n")
    f.write('\n' + '=' * 90 + '\n')

print(f'Statistics saved to {stats_path}')